In [1]:
import pandas as pd
import spacy

In [2]:
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import os
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load("en_core_web_sm")

In [4]:
df = pd.read_csv('validated_requirements.csv')

In [5]:
def check_completeness(text):
    doc = nlp(text)
    
    # Criteria 1: Length check (Very short requirements are usually incomplete)
    if len(text.split()) < 5:
        return "Incomplete: Too short"
    
    # Criteria 2: Check for a Subject (Actor) and a Verb (Action)
    has_subject = any(token.dep_ in ('nsubj', 'nsubjpass') for token in doc)
    has_verb = any(token.pos_ == 'VERB' or token.pos_ == 'AUX' for token in doc)
    
    # Criteria 3: Check for "Shall" or "Should" (Standard requirement keywords)
    has_modal = any(token.lemma_.lower() in ['shall', 'should', 'must', 'will'] for token in doc)
    
    if not has_subject:
        return "Incomplete: Missing a subject (Who/What?)"
    if not has_verb:
        return "Incomplete: Missing an action (Does what?)"
    if not has_modal:
        return "Incomplete: Missing a requirement keyword (shall/must)"
    
    return "Complete"

In [7]:
df['completeness_status'] = df['requirement_sentence'].apply(check_completeness)

In [8]:
incomplete_reqs = df[df['completeness_status'] != "Complete"]

In [9]:
print("\n--- Completeness Report ---")
if incomplete_reqs.empty:
    print("All requirements passed the quality check!")
else:
    print(f"Found {len(incomplete_reqs)} incomplete or vague requirements:")
    display(incomplete_reqs[['requirement_sentence', 'completeness_status']])


--- Completeness Report ---
All requirements passed the quality check!


In [10]:
df.to_csv('final_checked_requirements.csv', index=False)
print("\nResults saved to 'final_checked_requirements.csv'")


Results saved to 'final_checked_requirements.csv'
